# Orbit Wars — Uçtan Uca Eğitim Akışı (Notebook Walkthrough)

Bu defter, **Orbit Wars** PPO pipeline'ını tek bir oturumda **gözlem inşasından PPO güncellemesine** kadar adım-adım çalıştırır ve her aşamayı kısa açıklamalarla anlatır.

## Mimari özet

```
env.observation
      │
      ▼
build_observation()  ────────►  Observation(planet_feats, fleet_feats, global_feats,
      │                                    edge_bias, masks)
      ▼
pack_observations()  ────────►  batch dict (torch tensors, padded P×P, F)
      │
      ▼
OrbitWarsPolicy.forward()  ──►  PolicyOutput(
      │                            src_logits  (B, P+1)        # NO-OP dahil
      │                            tgt_logits  (B, P, P)
      │                            ship_params (B, P, P, 2)    # Beta(α, β)  ← yeni
      │                            value       (B,)
      │                          )
      ▼
policy.act()  ───────────────►  {src, tgt, ship_value∈[0,1], is_noop, log_prob}
      │
      ▼
decode_action_with_launch()  ►  env action  + launch_info {ships, overflow_ships, raw_request}
      │                                                       ▲
      ▼                                                       │ continuous Beta + overflow
env.step([env_action])  ────►  next_obs, raw_reward, done     │
      │
      ▼
ProductionRewardTracker / PlanetDefenseTracker / FleetTracker
      │   capture, reinforcement (Sorun 1), clawback (Sorun 2),
      │   defense window, defense deficit shortfall, overflow penalty,
      │   airborne cost, planet_loss
      ▼
RolloutBuffer.add() + add_reward() + add_delayed_reward()
      │
      ▼
buffer.compute_gae()      # advantages, returns
      │
      ▼
ppo_update(policy, buffer)
      │
      ▼
yeni policy state  → bir sonraki episode
```

Bu defter **tek bir mini iteration**'ı gerçek kod yollarıyla çalıştıracak; her hücrenin altında ne yaptığı kısaca açıklanacak.

> **Not:** Bu notebook deposunun **kökünden** çalıştırılmak için yazılmıştır (`docs/` altından çalıştırırsanız ilk hücredeki `os.chdir` halleder).


## 1) Ortam ve sabit kurulum

İlk hücre proje köküne `chdir` yapar ve PyTorch determinizm seed'lerini sabitler. Hücreler sırasıyla çalıştırılmalıdır.

In [1]:
import os
import sys
from pathlib import Path

# Repo kökünü tespit et: notebook docs/ altından açıldıysa bir üst dizine geç.
REPO = Path.cwd()
if REPO.name == "docs":
    REPO = REPO.parent
os.chdir(REPO)
sys.path.insert(0, str(REPO))
print("CWD:", Path.cwd())

import numpy as np
import torch
torch.manual_seed(0); np.random.seed(0)

# Kaggle env aksi gürültüyü logger seviyesine çek.
import logging
logging.getLogger("kaggle_environments.envs.open_spiel_env.open_spiel_env").setLevel(logging.ERROR)
print("torch:", torch.__version__, "| device:", "cuda" if torch.cuda.is_available() else "cpu")

CWD: /home/doruk/Documents/Kaggle/Orbit_Wars


torch: 2.11.0+cu128 | device: cuda


## 2) Çevre — `kaggle_environments` üzerinden Orbit Wars

Tek bir env yaratıp ilk frame'in raw observation'ını inceleyelim. Bu raw dict; `obs_builder.build_observation()` onu modele vereceğimiz tensor formuna çevirir.

In [2]:
from kaggle_environments import make

env = make(
    "orbit_wars",
    configuration={"num_agents": 2, "episodeSteps": 60},
    debug=False,
)
env.reset(num_agents=2)
# Orbit Wars planet'leri **ilk step**'te yerleştiriyor; rollout.py de aynısını yapar.
state = env.step([[] for _ in range(2)])
raw_obs = state[0]["observation"]            # P0 perspektifinden ham gözlem
omega = raw_obs.get("angular_velocity", 0.0) or 0.0

print("Top-level anahtarlar :", sorted(raw_obs.keys()))
print("# planets            :", len(raw_obs["planets"]))
print("# fleets (turn=1)    :", len(raw_obs["fleets"]))
print("angular_velocity     :", omega)
print("İlk gezegen satırı   :", raw_obs["planets"][0])
print("  (sırasıyla: id, owner, x, y, radius, ships, production)")

[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO: Successfully loaded OpenSpiel environments: 16.


[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_backgammon


[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_checkers


[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_chess


[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_connect_four


[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_gin_rummy


... (10 outputs left out)

[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_python_repeated_pokerkit


[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO: OpenSpiel games skipped: 1.


[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    snake


Top-level anahtarlar : ['angular_velocity', 'comet_planet_ids', 'comets', 'fleets', 'initial_planets', 'next_fleet_id', 'planets', 'player', 'remainingOverageTime', 'step']
# planets            : 28
# fleets (turn=1)    : 0
angular_velocity     : 0.03197383070936925
İlk gezegen satırı   : [0, -1, 62.852811038889, 97.11865507292579, 1.6931471805599454, 54, 2]
  (sırasıyla: id, owner, x, y, radius, ships, production)


## 3) Observation İnşası — `obs_builder.build_observation`

`Observation` dataclass; her oyuncu için **kendi perspektifinden** üretilir:

| Alan | Şekil | İçerik |
|---|---|---|
| `planet_feats` | `(P_max, planet_dim≈34)` | Sahip flag'leri, gemi/üretim, yörünge, gelen filo bucketleri (kendi & düşman) |
| `fleet_feats`  | `(F_max, fleet_dim)` | Pozisyon, yön, ETA, sahiplik flag'leri |
| `global_feats` | `(global_dim,)` | Total gemi farkı, planet farkı, sun bilgisi vb. |
| `edge_bias`    | `(P_max, P_max)` | Pair-wise yumuşak distance bias |
| `planet_mask`  | `(P_max,)` bool | Padding'i ayırt eder |
| `source_mask`  | `(P_max,)` bool | "Bu turda buradan launch yapabilir miyim?" (sahibim, gemi var) |
| `planet_ids`   | `(P_max,)` int  | env-id ↔ slot eşlemesi |

Slot indeksleri (örn. `src_slot=0`) **env_id** değildir; `obs.planet_ids[src_slot]` ile env_id'ye dönülür. Padding slotları her turda yer değiştirebilir.

In [3]:
from obs_builder import (
    build_observation,
    P_MAX_DEFAULT, F_MAX_DEFAULT,
    PF_IDX_SHIPS, PF_IDX_INCOMING_SELF_B0, PF_IDX_INCOMING_ENEMY_B0,
    INCOMING_ETA_BUCKETS, SHIP_REQUEST_CAP,
)
from model import PLANET_FEATURE_DIM

P_MAX, F_MAX = P_MAX_DEFAULT, F_MAX_DEFAULT
obs0 = build_observation(raw_obs, player=0, angular_velocity=omega,
                         p_max=P_MAX, f_max=F_MAX,
                         planet_feature_dim=PLANET_FEATURE_DIM)

print("planet_feats   :", obs0.planet_feats.shape, obs0.planet_feats.dtype)
print("fleet_feats    :", obs0.fleet_feats.shape)
print("global_feats   :", obs0.global_feats.shape)
print("edge_bias      :", obs0.edge_bias.shape)
print("planet_mask    :", obs0.planet_mask.shape, "  aktif slot sayısı =", int(obs0.planet_mask.sum()))
print("source_mask    :", obs0.source_mask.shape, "  benim launch yapabileceğim slot sayısı =", int(obs0.source_mask.sum()))
print()
print("PF_IDX_SHIPS               =", PF_IDX_SHIPS)
print("PF_IDX_INCOMING_SELF_B0    =", PF_IDX_INCOMING_SELF_B0)
print("PF_IDX_INCOMING_ENEMY_B0   =", PF_IDX_INCOMING_ENEMY_B0)
print("INCOMING_ETA_BUCKETS (sn)  =", INCOMING_ETA_BUCKETS)
print("SHIP_REQUEST_CAP           =", SHIP_REQUEST_CAP, "  ← Beta head x SHIP_REQUEST_CAP = istenen gemi")
print()
print("İlk aktif slot ait gemi (normalize, /100 ölçekli):")
active = np.where(obs0.planet_mask)[0]
for s in active[:3]:
    print(f"  slot={s}  ships={obs0.planet_feats[s, PF_IDX_SHIPS]:.3f}"
          f"  source_ok={bool(obs0.source_mask[s])}  env_id={int(obs0.planet_ids[s])}")

planet_feats   : (44, 34) float32
fleet_feats    : (64, 11)
global_feats   : (14,)
edge_bias      : (109, 109, 6)
planet_mask    : (44,)   aktif slot sayısı = 28
source_mask    : (44,)   benim launch yapabileceğim slot sayısı = 1

PF_IDX_SHIPS               = 5
PF_IDX_INCOMING_SELF_B0    = 16
PF_IDX_INCOMING_ENEMY_B0   = 20
INCOMING_ETA_BUCKETS (sn)  = (5.0, 15.0, 30.0, inf)
SHIP_REQUEST_CAP           = 200   ← Beta head x SHIP_REQUEST_CAP = istenen gemi

İlk aktif slot ait gemi (normalize, /100 ölçekli):
  slot=0  ships=0.540  source_ok=False  env_id=0
  slot=1  ships=0.540  source_ok=False  env_id=1
  slot=2  ships=0.540  source_ok=False  env_id=2


## 4) `pack_observations` — batch tensor üret

Bir `list[Observation]` → `dict[str, torch.Tensor]`. PPO'da bir batch'te birden fazla zaman adımı / oyuncu olur; burada tek-elemanlı bir batch yapıyoruz.

In [4]:
from model import pack_observations

batch = pack_observations([obs0])
for k, v in batch.items():
    if isinstance(v, torch.Tensor):
        shape_str = str(tuple(v.shape))
        print(f"  {k:<14}{shape_str:<24} {v.dtype}")
    else:
        print(f"  {k:<14}(non-tensor) {type(v).__name__}")

  planet_feats  (1, 44, 34)              torch.float32
  fleet_feats   (1, 64, 11)              torch.float32
  global_feats  (1, 14)                  torch.float32
  edge_bias     (1, 109, 109, 6)         torch.float32
  planet_mask   (1, 44)                  torch.bool
  fleet_mask    (1, 64)                  torch.bool
  source_mask   (1, 44)                  torch.bool


## 5) Model — `GroupedPlanetOrbitWarsPolicy`

Kullandığımız ana mimari. Üç başlık vardır:

| Başlık | Çıktı | Yorum |
|---|---|---|
| `src_logits` | `(B, P+1)` | Hangi gezegenden launch? Son slot **NO-OP** (turn'ü pasla). |
| `tgt_logits` | `(B, P, P)` | Her source için hedef gezegen kategorisi. `tgt_logits[b, s, t]` = src=s iken tgt=t için logit. |
| `ship_params` | `(B, P, P, 2)` | **Sürekli Beta(α, β) parametreleri**, src×tgt çiftine bağlı. Örnek `ship_value ~ Beta(α, β) ∈ [0,1]`. Gerçek istek = `ship_value * SHIP_REQUEST_CAP`. |
| `value` | `(B,)` | Critic, V(s). |

**Beta head neden?** Eskiden 5 ayrık ship_bin (1, 5, 25, 125, 625) vardı. Sürekli head sayesinde:
- Granüler kontrol (ör. 87 gemi).
- Garrison'u aşan istekleri **overflow penalty** ile cezalandırabiliyoruz (öğrenmeyi daha zengin sinyalle besler).
- Tek bir scalar log_prob; PPO ratio hesabı temiz.

In [5]:
from model_grouped_planet import GroupedPlanetOrbitWarsPolicy

# Notebook hızı için bilinçli olarak küçük tutuyoruz.
# Gerçek eğitimde d_model=128, n_layers=4 kullanılıyor.
policy = GroupedPlanetOrbitWarsPolicy(
    p_max=P_MAX, f_max=F_MAX,
    planet_dim=PLANET_FEATURE_DIM,
    d_model=32, n_heads=2, n_layers=1,
)
policy.eval()
n_params = sum(p.numel() for p in policy.parameters())
print(f"Toplam parametre: {n_params:,}")

with torch.no_grad():
    out = policy(**batch)

print("\nForward çıktıları:")
print("  src_logits      :", tuple(out.src_logits.shape))
print("  tgt_logits      :", tuple(out.tgt_logits.shape))
print("  ship_params     :", tuple(out.ship_params.shape), "  ← (B, P, P, 2) = (alpha, beta)")
print("  value           :", tuple(out.value.shape), "→", out.value.item())
print("  aux_owner_logits:", tuple(out.aux_owner_logits.shape))

Toplam parametre: 37,080



Forward çıktıları:
  src_logits      : (1, 45)
  tgt_logits      : (1, 44, 44)
  ship_params     : (1, 44, 44, 2)   ← (B, P, P, 2) = (alpha, beta)
  value           : (1,) → 0.09291265904903412
  aux_owner_logits: (1, 44, 5)


### 5.1 Beta head: α, β → Beta dağılımı

`_beta_concentrations` fonksiyonu raw logits'i Beta(α, β) concentrations'a maple­r. Güncel formül:

```python
α = 1 + (MAX-1) · tanh(softplus(raw_α) / (MAX-1))      # MAX = 30
β = 1 + (MAX-1) · tanh(softplus(raw_β) / (MAX-1))
```

- `softplus + 1` α, β ≥ 1 garantiler ⇒ **unimodal** Beta (dağılım U-şekilli olmaz).
- `tanh ile [1, 30]'a soft-cap` ⇒ Beta'nın delta'ya çökmesini önler. Cap olmadan α, β → ∞ giderek
  log_prob keskinleşiyor ve PPO importance ratio $\exp(\Delta \log p)$ patlatıyordu (KL>100 patolojisi).

Aşağıda küçük bir örnekle Beta'nın ortalamasını ve örneğini görüyoruz:

In [6]:
from model import _beta_concentrations, _make_ship_dist

# İlk gerçek source slotunu seç ve örnek bir target seç.
src_slot = int(np.where(obs0.source_mask)[0][0])
# Aynı oyuncuya ait olmayan bir target slot
others = [t for t in np.where(obs0.planet_mask)[0]
          if t != src_slot and not obs0.source_mask[t]]
tgt_slot = int(others[0])

raw_params = out.ship_params[0, src_slot, tgt_slot]   # (2,)
alpha, beta = _beta_concentrations(raw_params)
dist = _make_ship_dist(raw_params)

print(f"src_slot={src_slot}  tgt_slot={tgt_slot}")
print(f"raw  α,β logits : {raw_params.tolist()}")
print(f"clamped α       : {alpha.item():.4f}")
print(f"clamped β       : {beta.item():.4f}")
print(f"Beta mean       : {(alpha/(alpha+beta)).item():.4f}")
print(f"Beta sample     : {dist.sample().item():.4f}   ← ship_value ∈ [0,1]")
print(f"İstenen gemi    : {dist.sample().item() * SHIP_REQUEST_CAP:.1f}  (ship_value * SHIP_REQUEST_CAP)")

src_slot=16  tgt_slot=0
raw  α,β logits : [-0.03594159334897995, 0.18651731312274933]
clamped α       : 1.6752
clamped β       : 1.7906
Beta mean       : 0.4834
Beta sample     : 0.7331   ← ship_value ∈ [0,1]
İstenen gemi    : 54.1  (ship_value * SHIP_REQUEST_CAP)


## 6) Action sampling — `policy.act`

`policy.act(out, deterministic=False)` her örnek için:
1. `Categorical(src_logits)` → src
2. `Categorical(tgt_logits[:, src])` → tgt (tgt_logits zaten geçersiz hedefleri maskeler)
3. `Beta(α, β) at (src, tgt)` → `ship_value`
4. NO-OP ise tgt/ship dalları **log_prob'a katkı yapmaz** (mask).

Toplam log-prob: `log p(src) + 1[!noop]·(log p(tgt|src) + log p(ship_value|src,tgt))`.

In [7]:
torch.manual_seed(7)
with torch.no_grad():
    action = policy.act(out, deterministic=False)

print("Sampled action (single env):")
for k, v in action.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k:<10}{tuple(v.shape)}  {v.dtype}  →  {v.tolist()}")

# evaluate_actions: act() ile tutarlı log_prob
ev = policy.evaluate_actions(out, action["src"], action["tgt"], action["ship_value"])
print("\nReproducibility kontrolü:")
print(f"  act log_prob       = {action['log_prob'].item():.6f}")
print(f"  evaluate log_prob  = {ev['log_prob'].item():.6f}")
print(f"  entropy            = {ev['entropy'].item():.4f}")

Sampled action (single env):
  src       (1,)  torch.int64  →  [16]
  tgt       (1,)  torch.int64  →  [7]
  ship_value(1,)  torch.float32  →  [0.8413698673248291]
  is_noop   (1,)  torch.bool  →  [False]
  log_prob  (1,)  torch.float32  →  [-3.615213394165039]

Reproducibility kontrolü:
  act log_prob       = -3.615213
  evaluate log_prob  = -3.615213
  entropy            = 3.2447


## 7) Action decode — `decode_action_with_launch` ve **overflow penalty**

`obs_builder.decode_action_with_launch(obs, src_slot, tgt_slot, ship_value, ...)`:
- src/tgt env-id'leri çıkarır (`obs.planet_ids`).
- `navigation.compute_launch(...)` ile **fizik tabanlı** launch açısı + gemi sayısı bulur (**istenen gemi** = `clamp(ship_value * SHIP_REQUEST_CAP, 0, src.ships)`).
- Geri dönen `launch_info` dict'i:
  - `ships`        — env'e gerçekten gönderilecek (≤ kaynaktaki gemi).
  - `raw_request`  — modelin istediği (`ship_value * SHIP_REQUEST_CAP`).
  - `overflow_ships` = `max(0, raw_request − available)`.
- Eğer kaynakta hiç gemi yoksa veya target ulaşılamazsa **`None`** döner → o (src, tgt) çifti turn'de tüketilmez, başka çift denenir.

`overflow_ships > 0` ise rollout reward step'inde **sürekli ship head'ı kendi disiplinini** öğrensin diye:

```
overflow_penalty = -ship_overflow_weight * overflow_ships * reward_scale
```

uygulanır (her ödül sisteminde aktif, sadece `production_simple`'a bağlı değildir).

In [8]:
from obs_builder import decode_action_with_launch
from feature_extraction import parse_planets

parsed = parse_planets(raw_obs)
src_slot = int(action["src"][0])
tgt_slot = int(action["tgt"][0])
ship_value = float(action["ship_value"][0])
is_noop = bool(action["is_noop"][0])
print(f"Sampled: src_slot={src_slot}, tgt_slot={tgt_slot}, ship_value={ship_value:.3f}, noop={is_noop}")

# NO-OP gelirse manuel olarak gerçek bir launch örnekliyoruz:
if is_noop or src_slot == policy.p_max:
    src_slot = int(np.where(obs0.source_mask)[0][0])
    others = [t for t in np.where(obs0.planet_mask)[0]
              if t != src_slot and not obs0.source_mask[t]]
    tgt_slot = int(others[0])
    ship_value = 0.95            # bilerek aşırı talep
    print(f"NO-OP gelmişti, manuel: src={src_slot}, tgt={tgt_slot}, ship_value={ship_value}")

decoded = decode_action_with_launch(
    obs0, src_slot, tgt_slot, ship_value,
    planets=parsed, angular_velocity=omega, no_op=False,
)
if decoded is None:
    print("compute_launch çözüm bulamadı (sun çarpışması/ETA?). Hücreyi tekrar çalıştırın.")
else:
    env_action, launch_info = decoded
    print("env_action  =", env_action, "  ([from_id, angle_rad, ships])")
    print("launch_info =", launch_info)
    print(f"\n→ ship_value*CAP = {ship_value*SHIP_REQUEST_CAP:.0f}, gerçek = {launch_info['ships']}, overflow = {launch_info['overflow_ships']:.1f}")

Sampled: src_slot=16, tgt_slot=7, ship_value=0.841, noop=False
compute_launch çözüm bulamadı (sun çarpışması/ETA?). Hücreyi tekrar çalıştırın.


## 8) Çevre adımı

`env.step([actions_p0, actions_p1])` her oyuncu için **bir liste** alır (turn başına çoklu launch). PPO rollout'u turn başına **8 launch'a kadar** dener; biz şimdi sadece bir tane gösterip env'i ilerleteceğiz.

In [9]:
# P0 yukarıdaki env_action'ı atsın, P1 bu turn pas geçsin (boş liste = NO-OP).
p0_actions = [env_action] if decoded is not None else []
p1_actions = []

states = env.step([p0_actions, p1_actions])
post_obs = states[0].observation

print("Reward (P0, P1):", states[0].reward, states[1].reward)
print("Done           :", states[0].status == "DONE")
print("# fleets after :", len(post_obs["fleets"]))
print("İlk filo (varsa):", post_obs["fleets"][0] if post_obs["fleets"] else "—")

Reward (P0, P1): 0 0
Done           : False
# fleets after : 0
İlk filo (varsa): —


## 9) `collect_episode` — tüm rollout pipeline'ı

Yukarıda tek launch'ı manuel yaptık; gerçek eğitimde bunların hepsi `rl/rollout.py::collect_episode` içinde döner:

1. **Decision loop** — Her turn ve her oyuncu için: source-mask varsa `policy(out)` çağır, geçerli (src, tgt) çifti seç, `decode_action_with_launch` ile launch info çıkar, **fleet_meta** kaydet (eta, ships, overflow_ships, source_defense_shortfall, …).
2. **Action multiplex** — Turn başına 8 launch'a kadar; her seferinde source mask güncellenir.
3. **`env.step`** — toplu launch listesi gider.
4. **Reward step** — `compute_step_reward` + tracker'lar:
   - `FleetTracker` → fleet airborne / capture / kill / waste
   - `PlanetDefenseTracker` → "Defense Window" planet loss kredi paylaşımı (depart vs hold)
   - `ProductionRewardTracker` → capture, neutral_gift, **reinforcement (Sorun 1)**, **clawback (Sorun 2)**
   - `PlanetLossTracker` → legacy, sadece `current` / `aggressive`
   - **Overflow penalty** — Beta head'ın garrison aşımını cezalandırır
   - **Defense deficit** — ETA-doğru shortfall: launch'tan sonra tüm gelen düşman + üretim simüle edilir; net açık varsa cezalandırılır.
5. **Buffer.add** — her PPO transition için (planet_feats, fleet_feats, src, tgt, ship_value, log_prob, value, reward, done).

Aşağıda küçük bir env ile **gerçek `collect_episode`** çağırıyoruz.

In [10]:
from rl.rollout import collect_episode
from rl.buffer import RolloutBuffer
from rl.rewards import build_reward_config

reward_cfg = build_reward_config(
    "production_simple",
    ship_overflow_weight=1.0,    # Beta head disiplin sinyali açık
)

buffer = RolloutBuffer()
stats = collect_episode(
    policy=policy,
    buffer=buffer,
    max_steps=40,        # notebook hızı için kısa episode
    num_agents=2,
    reward_cfg=reward_cfg,
    seed=42,
    device=torch.device("cpu"),
)

print("Episode stats (EpisodeStats dataclass):")
print(f"  length             {stats.length}")
print(f"  winner             {stats.winner}")
print(f"  rewards (P0, P1)   {stats.rewards}")
print(f"  n_launches_p0/p1   {stats.n_launches_p0} / {stats.n_launches_p1}")

print(f"\nBuffer doluluk: {len(buffer.data)} transition")
print("İlk transition reward       :", buffer.data[0].reward)
print("İlk transition src/tgt/sv/no:",
      buffer.data[0].src, buffer.data[0].tgt,
      f"{buffer.data[0].ship_value:.3f}", buffer.data[0].is_noop)

Episode stats (EpisodeStats dataclass):
  length             40
  winner             -1
  rewards (P0, P1)   [-1.0, -1.0]
  n_launches_p0/p1   40 / 40

Buffer doluluk: 80 transition
İlk transition reward       : -0.23982862914973438
İlk transition src/tgt/sv/no: 20 8 0.398 False


## 10) Ödül kanallarını ayrıştır — `analyze_rewards.DiagnosticRolloutBuffer`

`rl.analyze_rewards` modülü, `RolloutBuffer.add_reward` çağrısını **stack frame** üzerinden hangi koddan geldiğine bakarak kanala atar. Bu sayede tek skaler `reward` toplamının arkasında **hangi sinyal ne kadar katkı verdiğini** görürüz.

`_COMPONENT_RANGES` `rl/rollout.py` içindeki satır aralıklarıyla tanımlıdır; kod taşındığında buranın güncel kalması gerekir.

In [11]:
from rl.analyze_rewards import DiagnosticRolloutBuffer, _COMPONENT_RANGES

print("Tanımlı reward kanalları (rollout.py satır aralığı → ad):")
for (lo, hi), name in _COMPONENT_RANGES:
    print(f"  L{lo:>4}-{hi:<4}  {name}")

diag_buf = DiagnosticRolloutBuffer()
collect_episode(policy=policy, buffer=diag_buf,
                max_steps=40, num_agents=2,
                reward_cfg=reward_cfg, seed=43,
                device=torch.device("cpu"))

print("\nEpisode boyunca her kanalın toplam sinyal büyüklüğü (|sum|):")
totals = {name: sum(r for _, r in items) for name, items in diag_buf.components.items()}
for name in sorted(totals, key=lambda n: -abs(totals[n])):
    n_events = len(diag_buf.components[name])
    print(f"  {name:<18}  Σreward = {totals[name]:+.5f}   (#events = {n_events})")

Tanımlı reward kanalları (rollout.py satır aralığı → ad):
  L 798-815   fleet_combat
  L 828-832   step_reward
  L 882-888   airborne
  L 895-902   overflow
  L 909-912   defense_deficit
  L 970-980   defense_hold
  L1005-1015  planet_defense
  L1018-1029  capture
  L1030-1041  neutral_gift
  L1042-1056  reinforcement
  L1057-1070  capture_clawback
  L1071-1085  planet_loss



Episode boyunca her kanalın toplam sinyal büyüklüğü (|sum|):
  overflow            Σreward = -12.52348   (#events = 81)
  unknown             Σreward = -8.16696   (#events = 175)
  planet_loss         Σreward = +2.27900   (#events = 8)
  planet_defense      Σreward = -2.00000   (#events = 81)


## 11) GAE — Generalized Advantage Estimation

`buffer.compute_gae(last_value, gamma, lam)` her transition için:

```
δ_t = r_t + γ·V(s_{t+1})·(1-done) - V(s_t)
A_t = δ_t + γ·λ·(1-done)·A_{t+1}
R_t = A_t + V(s_t)                     ← target for value head
```

Self-play'de **iki oyuncunun transition'ları aynı buffer'da iç içe** olur; `compute_gae` her oyuncuyu kendi alt zincirinde işler (player id ile gruplar).

In [12]:
buffer.compute_gae(last_value=0.0, gamma=0.99, lam=0.95)
adv = np.array(buffer._advantages)
ret = np.array(buffer._returns)
val = np.array([t.value for t in buffer.data])

print(f"#transitions   : {len(buffer.data)}")
print(f"advantages stats: mean={adv.mean():+.4f}  std={adv.std():.4f}  min={adv.min():+.4f}  max={adv.max():+.4f}")
print(f"returns stats   : mean={ret.mean():+.4f}  std={ret.std():.4f}")
print(f"values   stats  : mean={val.mean():+.4f}  std={val.std():.4f}")

# Cüzi bir grafik: ilk 100 adımın advantage / value
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 1, figsize=(10, 3.5))
N = min(120, len(buffer.data))
ax.plot(adv[:N], label="advantage")
ax.plot(ret[:N], label="return", alpha=0.7)
ax.plot(val[:N], label="value (predicted)", alpha=0.7)
ax.axhline(0, color="k", lw=0.5)
ax.set_title("İlk transition'larda GAE çıktıları")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

#transitions   : 80
advantages stats: mean=-3.0981  std=1.1070  min=-5.1572  max=-1.0886
returns stats   : mean=-3.0998  std=1.0875
values   stats  : mean=-0.0017  std=0.0347


/tmp/ipykernel_107870/2851813019.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 12) PPO update — `rl/ppo.py::ppo_update`

Standart **clipped surrogate** + **value loss** + **entropy bonus** + **KL early-stop**:

$$
L_{policy} = -\mathbb{E}\Big[\min\big(r(\theta) A,\; \mathrm{clip}(r(\theta), 1\!-\!\varepsilon, 1\!+\!\varepsilon)\, A\big)\Big]
$$

$$
L_{value} = \tfrac{1}{2}\,\mathbb{E}[(V_\theta(s) - R)^2]
$$

$$
L = L_{policy} + c_v L_{value} - c_H \mathcal{H}[\pi]
$$

Her minibatch için `policy.evaluate_source_actions(out, src, tgt, ship_value, is_noop)` çağrısıyla
**yeni log_prob** ve **entropy** alınır; **eski** log_prob buffer'da saklı. Beta head'in entropisi
diferansiyeldir; categorical action ile toplanır.

**Yeni:**

- `target_kl=0.05` (default): bir inner epoch'un mean approx_kl'i target'ı aşarsa kalan epoch'lar
  atlanır (`epochs_run < epochs_total`). Trust-region güvencesi için kritik.
- `PPOStats` artık şunları da raporluyor: `ratio_q05/q50/q95` (importance ratio quantile'ları),
  `ship_alpha_mean / ship_beta_mean` (Beta concentration sağlığı), `epochs_run`.

In [13]:
from rl.ppo import ppo_update, PPOConfig
from dataclasses import asdict

# Optimizer hazırla
optim = torch.optim.Adam(policy.parameters(), lr=3e-4)
ppo_cfg = PPOConfig(
    clip_range=0.2, epochs=2, batch_size=64,
    entropy_coef=0.01, value_coef=0.5,
    target_kl=0.05,           # KL early-stop
)

policy.train()
stats = ppo_update(
    policy=policy, buffer=buffer, optimizer=optim, cfg=ppo_cfg,
    device=torch.device("cpu"),
)

print("PPO update metrikleri (PPOStats dataclass):")
for k, v in asdict(stats).items():
    if isinstance(v, float):
        print(f"  {k:<20}{v:+.5f}")
    elif isinstance(v, dict):
        print(f"  {k:<20}{len(v)} module grad norms (örn: {next(iter(v.items()), '—')})")
    else:
        print(f"  {k:<20}{v}")

PPO update metrikleri (PPOStats dataclass):
  policy_loss         +0.00976
  value_loss          +5.15932
  entropy             +3.49922
  approx_kl           +0.00099
  clip_fraction       +0.00000
  n_updates           4
  last_grad_norms     9 module grad norms (örn: ('encoder', 1.6115309726966682))
  ratio_q05           +0.98981
  ratio_q50           +1.00806
  ratio_q95           +1.04013
  ship_alpha_mean     +1.67814
  ship_beta_mean      +1.74658
  epochs_run          2


## 13) Mini eğitim döngüsü — birkaç iter

`rl/train.py::train()`'in **küçültülmüş bir kopyası**: 5 iterasyon, 2 episode/iter, küçük model. Sadece akışın baştan sona çalıştığını ve metriklerin dalgalanma gösterdiğini gözlemlemek için.

In [14]:
history = {"polL": [], "valL": [], "H": [], "kl": [], "ep_len": [], "p0_reward": []}

# Notebook için kasıtlı çok küçük: gerçek eğitim CLI'dan yapılır.
ITERATIONS = 2
EPISODES_PER_ITER = 1
MAX_TURNS = 40

for it in range(ITERATIONS):
    buf = RolloutBuffer()
    ep_lens, p0_r = [], []
    for ep in range(EPISODES_PER_ITER):
        s = collect_episode(policy=policy, buffer=buf,
                            max_steps=MAX_TURNS, num_agents=2,
                            reward_cfg=reward_cfg,
                            seed=1000 + 31 * it + ep,
                            device=torch.device("cpu"))
        ep_lens.append(s.length)
        p0_r.append(s.reward_p0)

    buf.compute_gae(last_value=0.0, gamma=0.99, lam=0.95)
    m = ppo_update(policy=policy, buffer=buf, optimizer=optim, cfg=ppo_cfg,
                   device=torch.device("cpu"))

    history["polL"].append(m.policy_loss)
    history["valL"].append(m.value_loss)
    history["H"].append(m.entropy)
    history["kl"].append(m.approx_kl)
    history["ep_len"].append(np.mean(ep_lens))
    history["p0_reward"].append(np.mean(p0_r))
    print(f"[it {it}] len̄={np.mean(ep_lens):4.1f}  P0r̄={np.mean(p0_r):+5.2f}  "
          f"polL={m.policy_loss:+.4f}  valL={m.value_loss:.4f}  "
          f"H={m.entropy:.3f}  KL={m.approx_kl:.4f}")

[it 0] len̄=40.0  P0r̄=-1.00  polL=+0.0390  valL=14.7967  H=3.092  KL=0.0010


[it 1] len̄=40.0  P0r̄=-1.00  polL=+0.0007  valL=7.7431  H=3.240  KL=0.0001


In [15]:
fig, axes = plt.subplots(2, 2, figsize=(11, 6))
axes[0, 0].plot(history["polL"], "o-"); axes[0, 0].set_title("policy_loss"); axes[0, 0].grid(alpha=0.3)
axes[0, 1].plot(history["valL"], "o-", color="tab:orange"); axes[0, 1].set_title("value_loss"); axes[0, 1].grid(alpha=0.3)
axes[1, 0].plot(history["H"], "o-", color="tab:green"); axes[1, 0].set_title("entropy"); axes[1, 0].grid(alpha=0.3)
axes[1, 1].plot(history["kl"], "o-", color="tab:red"); axes[1, 1].set_title("approx KL"); axes[1, 1].grid(alpha=0.3)
for ax in axes.flat:
    ax.set_xlabel("iter")
plt.tight_layout(); plt.show()

print("Bitiş özeti:")
print(f"  ortalama episode uzunluğu (it sonu): {history['ep_len'][-1]:.1f}")
print(f"  ortalama P0 raw reward (it sonu)    : {history['p0_reward'][-1]:+.3f}")

Bitiş özeti:
  ortalama episode uzunluğu (it sonu): 40.0
  ortalama P0 raw reward (it sonu)    : -1.000


/tmp/ipykernel_107870/1053072893.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 14) Scripted opponent ile eğitim (opsiyonel)

`opponent_indices=[1]` ve `scripted_opponent_cfg` parametreleri verilirse P1'i `CaptureEfficiencyAgent` (deterministik scripted bot) oynar; PPO sadece P0'a optimize edilir. Curriculum başlangıcı için faydalıdır.

In [16]:
from scripted_strategy import ScriptedConfig

buf_sc = RolloutBuffer()
stats_sc = collect_episode(
    policy=policy, buffer=buf_sc,
    max_steps=40, num_agents=2,
    reward_cfg=reward_cfg, seed=999,
    device=torch.device("cpu"),
    opponent_indices={1},                        # P1 scripted
    scripted_opponent_cfg=ScriptedConfig(),      # default davranış
)
print(f"Scripted opponent ile {len(buf_sc.data)} PPO transition (sadece P0) toplandı.")
print("İlk transition: player =", buf_sc.data[0].player, "  reward =", buf_sc.data[0].reward)
print("Tüm transition'lar P0'a ait mi?",
      all(t.player == 0 for t in buf_sc.data))

Scripted opponent ile 42 PPO transition (sadece P0) toplandı.
İlk transition: player = 0   reward = -0.027150124656195124
Tüm transition'lar P0'a ait mi? True


## 15) Komuta satırı (gerçek eğitim)

Notebook'un kapsamı tek bir mini iter içindi. **Gerçek eğitim** için terminalden:

```bash
orbit_wars_venv/bin/python -m rl.train \
    --iterations 200 --episodes-per-iter 32 --max-episode-steps 600 \
    --num-agents 2 --model-variant grouped_planet \
    --d-model 128 --n-heads 4 --n-layers 4 \
    --reward-system production_simple \
    --ship-overflow-weight 1.0 \
    --opponent self_play \
    --checkpoint-every 5
```

Bayraklar (özet):

| Bayrak | Anlam |
|---|---|
| `--reward-system` | `current` / `aggressive` / `production_simple` |
| `--ship-overflow-weight` | Beta head garrison aşımı cezası |
| `--planet-loss-weight` | Defense window + defense deficit ölçeği |
| `--production-reinforcement-weight` (config) | Sorun 1 |
| `--production-capture-clawback-fraction` (config) | Sorun 2 |
| `--opponent {self_play, scripted}` | Karşı tarafı kim oynar |

## 16) İzleme — `python -m rl.watch <ckpt>`

Eğitim sonunda checkpoint'ı izlemek için:

```bash
orbit_wars_venv/bin/python -m rl.watch checkpoints/policy_iter00050.pt --opponent scripted --html replays/iter50.html
```

`rl/watch.py` Beta head ortalamasını (`α/(α+β)`) deterministik yeniden oynatma için kullanır.

---

**Bu notebook tek başına:** model + observation + rollout + reward + buffer + GAE + PPO update halkalarını gerçek kod yollarıyla çalıştırır. Her hücre bir aşamayı doğrular ve tipik çıktı şekillerini gösterir. Gerçek eğitim, parametre büyüklüğü ve uzun episode sayısı gerektirdiğinden CLI üzerinden yapılır.